In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils import shuffle

from skimage.feature import hog

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
import pandas as pd
import numpy as np

from sklearn.utils import shuffle

csv_path = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv"

df = pd.read_csv(csv_path)

print(df.shape)

emotion_names = [
    'Angry',
    'Disgust',
    'Fear',
    'Happy',
    'Sad',
    'Surprise',
    'Neutral'
]

data = []
labels = []

for i in range(len(df)):

    pixels = np.array(
        df.loc[i, ' pixels'].split(),
        dtype=np.uint8
    )

    img = pixels.reshape(48,48)

    data.append(img)

    labels.append(df.loc[i, 'emotion'])

data = np.array(data)
labels = np.array(labels)

print("Original:", data.shape)

data, labels = shuffle(
    data,
    labels,
    random_state=42
)

data = data[:5000]
labels = labels[:5000]

print("Subset:", data.shape)

(35887, 3)
Original: (35887, 48, 48)
Subset: (5000, 48, 48)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(3500, 48, 48)
(1500, 48, 48)


In [5]:
from skimage.feature import hog

X_hog_train = []

for img in X_train:

    feat = hog(
        img,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2)
    )

    X_hog_train.append(feat)

X_hog_train = np.array(X_hog_train)

X_hog_test = []

for img in X_test:

    feat = hog(
        img,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2)
    )

    X_hog_test.append(feat)

X_hog_test = np.array(X_hog_test)

print(X_hog_train.shape)

(3500, 900)


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score

scaler_hog = StandardScaler()

X_hog_train_s = scaler_hog.fit_transform(X_hog_train)
X_hog_test_s = scaler_hog.transform(X_hog_test)

hog_model = SVC(
    kernel='linear'
)

hog_model.fit(
    X_hog_train_s,
    y_train
)

y_pred_hog = hog_model.predict(
    X_hog_test_s
)

hog_acc = accuracy_score(
    y_test,
    y_pred_hog
)

hog_f1 = f1_score(
    y_test,
    y_pred_hog,
    average='macro'
)

print("HOG Accuracy:", hog_acc)
print("HOG F1:", hog_f1)

HOG Accuracy: 0.31466666666666665
HOG F1: 0.2871468684395504


In [7]:
from skimage.feature import local_binary_pattern

X_lbp_train = []

for img in X_train:

    lbp = local_binary_pattern(
        img,
        P=8,
        R=1,
        method='uniform'
    )

    hist,_ = np.histogram(
        lbp.ravel(),
        bins=10,
        range=(0,10)
    )

    hist = hist/(hist.sum()+1e-6)

    X_lbp_train.append(hist)

X_lbp_train = np.array(X_lbp_train)

X_lbp_test = []

for img in X_test:

    lbp = local_binary_pattern(
        img,
        P=8,
        R=1,
        method='uniform'
    )

    hist,_ = np.histogram(
        lbp.ravel(),
        bins=10,
        range=(0,10)
    )

    hist = hist/(hist.sum()+1e-6)

    X_lbp_test.append(hist)

X_lbp_test = np.array(X_lbp_test)

In [8]:
scaler_lbp = StandardScaler()

X_lbp_train_s = scaler_lbp.fit_transform(X_lbp_train)
X_lbp_test_s = scaler_lbp.transform(X_lbp_test)

lbp_model = SVC(
    kernel='linear'
)

lbp_model.fit(
    X_lbp_train_s,
    y_train
)

y_pred_lbp = lbp_model.predict(
    X_lbp_test_s
)

lbp_acc = accuracy_score(
    y_test,
    y_pred_lbp
)

lbp_f1 = f1_score(
    y_test,
    y_pred_lbp,
    average='macro'
)

print("LBP Accuracy:", lbp_acc)
print("LBP F1:", lbp_f1)

LBP Accuracy: 0.24733333333333332
LBP F1: 0.06472960683203396


In [9]:
import cv2
from sklearn.cluster import MiniBatchKMeans

sift = cv2.SIFT_create()

train_desc = []
all_desc = []

for img in X_train:

    kp, desc = sift.detectAndCompute(
        img.astype(np.uint8),
        None
    )

    train_desc.append(desc)

    if desc is not None:
        all_desc.extend(desc)

all_desc = np.array(all_desc)

print(all_desc.shape)

(82733, 128)


In [10]:
num_clusters = 300

kmeans = MiniBatchKMeans(
    n_clusters=num_clusters,
    random_state=42,
    batch_size=500
)

kmeans.fit(all_desc)

print("Vocabulary Ready")

Vocabulary Ready


In [11]:
X_sift_train = []

for desc in train_desc:

    hist = np.zeros(num_clusters)

    if desc is not None:

        words = kmeans.predict(desc)

        for w in words:
            hist[w] += 1

    hist = hist/(hist.sum()+1e-6)

    X_sift_train.append(hist)

X_sift_train = np.array(X_sift_train)

In [12]:
X_sift_test = []

for img in X_test:

    kp, desc = sift.detectAndCompute(
        img.astype(np.uint8),
        None
    )

    hist = np.zeros(num_clusters)

    if desc is not None:

        words = kmeans.predict(desc)

        for w in words:
            hist[w] += 1

    hist = hist/(hist.sum()+1e-6)

    X_sift_test.append(hist)

X_sift_test = np.array(X_sift_test)

In [13]:
scaler_sift = StandardScaler()

X_sift_train_s = scaler_sift.fit_transform(X_sift_train)
X_sift_test_s = scaler_sift.transform(X_sift_test)

sift_model = SVC(
    kernel='linear'
)

sift_model.fit(
    X_sift_train_s,
    y_train
)

y_pred_sift = sift_model.predict(
    X_sift_test_s
)

sift_acc = accuracy_score(
    y_test,
    y_pred_sift
)

sift_f1 = f1_score(
    y_test,
    y_pred_sift,
    average='macro'
)

print("SIFT Accuracy:", sift_acc)
print("SIFT F1:", sift_f1)

SIFT Accuracy: 0.23066666666666666
SIFT F1: 0.2011429202062133


In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

hog_cv = cross_val_score(
    SVC(kernel='linear'),
    X_hog_train_s,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("HOG Mean:", hog_cv.mean())
print("HOG Std :", hog_cv.std())

HOG Mean: 0.32971428571428574
HOG Std : 0.015343681808111501


In [15]:
lbp_cv = cross_val_score(
    SVC(kernel='linear'),
    X_lbp_train_s,
    y_train,
    cv=cv,
    scoring='accuracy'
)

In [16]:
sift_cv = cross_val_score(
    SVC(kernel='linear'),
    X_sift_train_s,
    y_train,
    cv=cv,
    scoring='accuracy'
)